# SuperLightNet — BraTS 2023 | M0 Training Only

**Objective:** Retrain the **original SuperLightNet** on BraTS 2023 GLI and resume the verified **iteration-500** checkpoint toward **40,000 iterations**.

This notebook is for **training only**. It does not modify the architecture or perform evaluation. Run cells in order in a **GPU Colab runtime**. Checkpoint files are stored in Google Drive. **Do not run a fresh training command without `--resume`** when continuing the official experiment.

**Configuration:** seed 42 · batch size 1 · AdamW · LR 0.001 · weight decay 0.01 · 40,000-iteration schedule · save every 500 iterations · 875 training subjects.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Check GPU and paths

In [2]:
from pathlib import Path
import torch

BASE = Path('/content/drive/MyDrive/BrainTumorSegmentation')
DRIVE_PROJECT = BASE / 'SuperLightNet'
ARCHIVE = BASE / 'SuperLightNet.zip'
PROJECT = Path('/content/SuperLightNet')
CHECKPOINT = DRIVE_PROJECT / 'checkpoints/M0_official_seed42/latest.pth'
DATA_ROOT = PROJECT / 'datasets/raw/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData'

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Archive:', ARCHIVE.exists())
print('Checkpoint:', CHECKPOINT.exists())
assert torch.cuda.is_available(), 'Select a GPU runtime before training.'
assert ARCHIVE.is_file(), 'Project ZIP not found in Google Drive.'
assert CHECKPOINT.is_file(), 'Official resume checkpoint not found. Do not start from scratch.'

CUDA: True
GPU: Tesla T4
Archive: True
Checkpoint: True


## 3. Restore project from Drive ZIP (only if missing)

Extraction can take several minutes and requires free Colab disk space. This does not overwrite the saved Drive checkpoint.

In [3]:
import zipfile

required = [PROJECT / 'Jnetworks/superlightnet.py', PROJECT / 'brats2023_dataset.py', DATA_ROOT]
if not all(p.exists() for p in required):
    print('Restoring project from ZIP...')
    with zipfile.ZipFile(ARCHIVE, 'r') as z:
        z.extractall('/content')
else:
    print('Project and dataset already present; skipping extraction.')

assert DATA_ROOT.is_dir(), 'BraTS 2023 data directory missing.'
print('Subject folders:', len(list(DATA_ROOT.glob('BraTS-GLI-*'))))

Project and dataset already present; skipping extraction.
Subject folders: 1251


## 4. Restore current training code and check dataset splits

In [4]:
import shutil

for filename in ('train_m0.py', 'resumable_sampler.py', 'rng_utils.py'):
    source = DRIVE_PROJECT / filename
    assert source.is_file(), f'Missing saved script: {source}'
    shutil.copy2(source, PROJECT / filename)
    compile((PROJECT / filename).read_text(encoding='utf-8'), filename, 'exec')
    print('Restored:', filename)

assert (PROJECT / 'brats2023_dataset.py').is_file()
assert (PROJECT / 'Jnetworks/superlightnet.py').is_file()

expected = {'train': 875, 'val': 125, 'test': 251}
for split, count in expected.items():
    path = PROJECT / 'splits/brats2023' / f'{split}.txt'
    assert path.is_file(), f'Missing {split} split'
    ids = [x.strip() for x in path.read_text().splitlines() if x.strip()]
    assert len(ids) == count, f'{split}: {len(ids)} != {count}'
    print(split, len(ids))

Restored: train_m0.py
Restored: resumable_sampler.py
Restored: rng_utils.py
train 875
val 125
test 251


## 5. Inspect the official checkpoint before resuming

In [5]:
import torch

ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
iteration = int(ckpt['iteration'])
assert int(ckpt['max_iterations']) == 40000
assert int(ckpt['seed']) == 42
for key in ('model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'rng_state'):
    assert key in ckpt, f'Missing checkpoint state: {key}'
print(f'Official checkpoint: iteration {iteration} / 40000')
print('Checkpoint size (MB):', round(CHECKPOINT.stat().st_size / 1024**2, 2))
assert 0 < iteration < 40000, 'Checkpoint is not a resumable in-progress experiment.'
del ckpt

Official checkpoint: iteration 500 / 40000
Checkpoint size (MB): 34.37


## 6. Resume the next training segment

**Default:** resume to the next multiple of 500 iterations (e.g. 500 → 1,000). The learning-rate horizon stays at **40,000**, and `--resume` restores the existing weights and optimizer state. **Run this cell only once per segment; wait for completion.**

In [6]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "monai==1.3.0"
])

print("\nMONAI installation completed!")


MONAI installation completed!


In [7]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "uninstall",
    "-y",
    "monai"
])

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "monai>=1.5,<1.6"
])

print("MONAI installation completed.")

MONAI installation completed.


In [8]:
import sys
import torch
import monai

sys.path.insert(0, "/content/SuperLightNet")

from Jnetworks.superlightnet import NormalU_Net

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("MONAI:", monai.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("SuperLightNet import: SUCCESS")

Python: 3.13.15
PyTorch: 2.11.0+cu130
MONAI: 1.5.2
CUDA available: True
GPU: Tesla T4
SuperLightNet import: SUCCESS


In [10]:
import torch
import sys

sys.path.insert(0, "/content/SuperLightNet")

from Jnetworks.superlightnet import NormalU_Net

checkpoint_path = (
    "/content/drive/MyDrive/BrainTumorSegmentation/"
    "SuperLightNet/checkpoints/M0_official_seed42/latest.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Saved iteration:", checkpoint["iteration"])
print("Training target:", checkpoint["max_iterations"])

model = NormalU_Net(depths_unidirectional="small")

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

print("Model weights loaded successfully!")
print("Parameters:", sum(p.numel() for p in model.parameters()))

Saved iteration: 500
Training target: 40000
Model weights loaded successfully!
Parameters: 2972492


In [12]:
from pathlib import Path

rng_file = Path("/content/SuperLightNet/rng_utils.py")

print(rng_file.read_text())


import random
import numpy as np
import torch


def capture_rng_state():
    """Capture all relevant random-number-generator states."""
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
    }


def restore_rng_state(state):
    """Restore previously captured random-number-generator states."""
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())

    if state["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])



In [13]:
from pathlib import Path

rng_file = Path("/content/SuperLightNet/rng_utils.py")

code = '''
import random
import numpy as np
import torch


def capture_rng_state():
    """Capture all relevant random-number-generator states."""
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
    }


def restore_rng_state(state):
    """Restore previously captured random-number-generator states."""
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])

    cpu_state = torch.as_tensor(
        state["torch"], dtype=torch.uint8
    ).cpu()
    torch.set_rng_state(cpu_state)

    if state["cuda"] is not None and torch.cuda.is_available():
        cuda_states = [
            torch.as_tensor(s, dtype=torch.uint8).cpu()
            for s in state["cuda"]
        ]
        torch.cuda.set_rng_state_all(cuda_states)
'''

rng_file.write_text(code)

print("RNG restoration code updated successfully.")

RNG restoration code updated successfully.


In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT = Path("/content/SuperLightNet")

command = [
    sys.executable,
    "-u",
    "train_m0.py",
    "--seed", "42",
    "--batch-size", "1",
    "--lr", "0.001",
    "--weight-decay", "0.01",
    "--max-iterations", "40000",
    "--num-workers", "0",
    "--save-every", "500",
    "--stop-after", "4000",
    "--resume",
]

print("Resuming M0 training: iteration 500 → 1000", flush=True)

with subprocess.Popen(
    command,
    cwd=str(PROJECT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)

    exit_code = process.wait()

print("\nTraining exit code:", exit_code)

Resuming M0 training: iteration 500 → 1000
I0000 00:00:1792031065.572302   30656 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
M0 — SuperLightNet Baseline
Device: cuda
Seed: 42
GPU: Tesla T4

Loading training dataset...
Training subjects: 875

Creating SuperLightNet...
Parameters: 2972492
Parameters (M): 2.972492
Resumed at iteration: 2000
RNG states restored.

Starting training...
Target iterations: 40000
Iteration 02010 | Loss: 2.219282 | LR: 0.00099378
Iteration 02020 | Loss: 0.701141 | LR: 0.00099372


## 7. Verify progress saved to Google Drive

A successful process exit alone does not prove that training advanced; inspect the saved checkpoint.

In [15]:
from pathlib import Path
import torch
import shutil

drive_project = Path(
    "/content/drive/MyDrive/BrainTumorSegmentation/SuperLightNet"
)

checkpoint_path = (
    drive_project / "checkpoints/M0_official_seed42/latest.pth"
)

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Saved iteration:", ckpt["iteration"])
print("Target:", ckpt["max_iterations"])

assert ckpt["iteration"] == 1000

# Preserve our RNG compatibility fix in Google Drive.
shutil.copy2(
    "/content/SuperLightNet/rng_utils.py",
    drive_project / "rng_utils.py"
)

print("Checkpoint verified.")
print("RNG fix backed up to Google Drive.")

Saved iteration: 1000
Target: 40000
Checkpoint verified.
RNG fix backed up to Google Drive.


## Continue later

When GPU time is available again, reopen this notebook and run cells **1–7**. The resume cell reads the **current** checkpoint iteration automatically and advances by up to 500 iterations.

**Team handoff:** Share the source code, dataset split files, and checkpoint separately from the BraTS dataset. This notebook uses paths under the owner's Google Drive; teammates must copy files into those paths or adapt the constants **and the hard-coded checkpoint directory in `train_m0.py`** before running. Do not let multiple people overwrite the same official checkpoint.

**Caveat:** Checkpoint saving/resume has been tested, but exact equivalence between interrupted and uninterrupted training has not yet been established. Keep that limitation in the experiment notes.